### aim: 

### date: 

In [8]:
from IPython.display import HTML

HTML('''<script>
code_show=true; 
function code_toggle() {
 if (code_show){
 $('div.input').hide();
 } else {
 $('div.input').show();
 }
 code_show = !code_show
} 
$( document ).ready(code_toggle);
</script>
<form action="javascript:code_toggle()"><input type="submit" value="Click here to toggle on/off the raw code."></form>''')


In [9]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])?  y


In [10]:
import numpy as np
from cmocean import cm
import cartopy as cp
import cartopy.crs as ccrs
import netCDF4 as nc
import matplotlib.pyplot as plt
import xarray as xr

%matplotlib inline
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')
import cartopy.feature as cfeature
from importlib import reload
import matplotlib.path as mpath
import glob
import pickle
import pandas as pd
import seawater
import time
plt.rcParams.update({'font.size': 13})
font = {'family' : 'normal',
'weight' : 'normal',
'size'   : 13}
plt.rcParams['text.usetex'] = True
plt.rc('font', **font)

import sys
sys.path.append('/gpfs/home/mep22dku/scratch/SOZONE/UTILS')
import snippets as sp
reload(sp)

<module 'snippets' from '/gpfs/home/mep22dku/scratch/SOZONE/UTILS/snippets.py'>

In [11]:
dir(sp)

['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'sniplist',
 'tdf',
 'tfig',
 'tmask',
 'tmetro',
 'tmodi',
 'txr',
 'tylist']

### calculate AMOC

In [12]:
def get_max_amoc(tr, tyr, resultsdir = '/gpfs/home/mep22dku/scratch/TiMBER-runsets/visualize-runs/data/'):
    
    #resultsdir = '/gpfs/home/mep22dku/scratch/AMOC-PLANKTOM/ARIA-runset/data/'
    #resultsdir = '/gpfs/data/greenocean/software/resources/CDFTOOLS/MOCresults/'
    
    outputFile = f'{resultsdir}/{tr}_{tyr}-AMOC.nc'
    #print(outputFile)
    
    #ty = f'/gpfs/home/mep22dku/cdftools/MOCresults/{tr}_1m_{tyr}0101*MOC.nc'
    ty = f'/gpfs/data/greenocean/software/resources/CDFTOOLS/MOCresults/{tr}_1m_{tyr}0101*MOC.nc'
    t2 = glob.glob(ty)
    moc_dataset = xr.open_dataset(t2[0])

    atl_at_26 = np.squeeze(moc_dataset.zomsfatl.sel(y=94).values)
    tshape = np.shape(atl_at_26)
    len_ts = tshape[0]

    max_atl = np.zeros(len_ts)

    for i in range(0,len(max_atl)):
        max_atl[i] = np.nanmax(atl_at_26[i,:])
        
    nicetime = moc_dataset.indexes['time_counter'].to_datetimeindex()
    
    data_vars = {'AMOC':(['time_counter'], max_atl,
    {'units': 'Sv',
    'long_name':'Atlantic Meridional Overturning Circulation, max of streamfunction at 26N'}),
    }
    # define coordinates
    coords = {'time_counter': (['time_counter'], nicetime)}
    # define global attributes
    attrs = {'made in':'AMOC-PLANKTOM/ARIA-runset/yearly_summary.ipynb',
    }
    ds = xr.Dataset(data_vars=data_vars,
    coords=coords,
    attrs=attrs)
    try:
        ds.to_netcdf(outputFile)
    except:
        print('fail2save')
    
    return max_atl




In [13]:
mods = ['TOM12_TJ_A5BB','TOM12_TJ_C5BB', 'TOM12_TJ_A5H3', 'TOM12_TJ_A5H6',\
       'TOM12_TJ_25A0','TOM12_TJ_25C0']
# mods = ['TOM12_TJ_A5H3']

ex = False

yr1 = 2024; yr2 = 2025

if ex:

    for m in mods:

            for yr in range(yr1,yr2):

                if yr%10 == 0: print(f'{m} {yr}')
                try:
                    get_max_amoc(m, yr)
                except:
                    print(f'not calculating {m} {yr}')


## get by-province mean, for any province, for any model/field

In [14]:
def get_mean(yr,model,dtyp,tvar,mask = 'glob',twod = False, ):
    
    tdir = '/gpfs/data/greenocean/software/runs/'
    resdir = './data/'
    tf = glob.glob(f'{tdir}{model}/ORCA2_1m_{yr}*{dtyp}*.nc')[0]
    w = xr.open_dataset(tf)
    tmask = \
    xr.open_dataset('/gpfs/home/mep22dku/scratch/SOZONE/UTILS/mesh_mask3pt6_nicedims.nc')
    if twod == False:
        if mask == 'glob': mymask = tmask['vol']
        else: mymask = tmask['vol'] * tmask[mask]
        res = w[tvar].weighted(mymask).mean(dim = ['y','x'])
    else:
        if mask == 'glob': mymask = tmask['csize']
        else: mymask = tmask['csize'] * tmask[mask]
        res = w[tvar].weighted(mymask).mean(dim = ['y','x'])
        

    res = res.assign_coords(time_counter=res.indexes['time_counter'].to_datetimeindex())
    res['time_counter'].encoding.clear()  # Prevent CFTime encoding
    sn = f'{model}_{yr}_{tvar}_{mask}.nc'
    #print(sn)

    
    try:
        res.to_netcdf(f'{resdir}{sn}')
    except:
        print(f'failed 2 save {sn}')
    return res


yr = 1941
model = 'TOM12_TJ_AA00'
dtyp ='diad'
tvar = 'Cflx'
#w = get_mean(yr,model,dtyp,tvar,mask = 'A1',twod = True)

In [ ]:
ex = True

mods = ['TOM12_TJ_A5BB','TOM12_TJ_C5BB', 'TOM12_TJ_A5H3', 'TOM12_TJ_A5H6',\
       'TOM12_TJ_25A0','TOM12_TJ_25C0']
masks = ['glob','A1','A2']
#EXP
yr1 = 1940; yr2 = 2025

if ex:

    for m in mods:
        for ma in masks:
            for yr in range(yr1,yr2):
                d2 = True; dtyp = 'diad'; tvar = 'Cflx'
                if yr%10 == 0: print(f'{m} {ma} {yr} {tvar}')
                get_mean(yr,m,dtyp,tvar,ma,twod = d2)

                d2 = True; dtyp = 'diad'; tvar = 'PPINT'
                if yr%10 == 0: print(f'{m} {ma} {yr} {tvar}')
                get_mean(yr,m,dtyp,tvar,ma,twod = d2)

                d2 = True; dtyp = 'grid_T'; tvar = 'sos'
                if yr%10 == 0: print(f'{m} {ma} {yr} {tvar}')
                get_mean(yr,m,dtyp,tvar,ma,twod = d2)

                d2 = False; dtyp = 'diad'; tvar = 'EXP'
                if yr%10 == 0: print(f'{m} {ma} {yr} {tvar}')
                get_mean(yr,m,dtyp,tvar,ma,twod = d2)



TOM12_TJ_A5BB glob 1940 Cflx
TOM12_TJ_A5BB glob 1940 PPINT
TOM12_TJ_A5BB glob 1940 sos
TOM12_TJ_A5BB glob 1940 EXP
TOM12_TJ_A5BB glob 1950 Cflx
TOM12_TJ_A5BB glob 1950 PPINT
TOM12_TJ_A5BB glob 1950 sos
TOM12_TJ_A5BB glob 1950 EXP
TOM12_TJ_A5BB glob 1960 Cflx
TOM12_TJ_A5BB glob 1960 PPINT
TOM12_TJ_A5BB glob 1960 sos
TOM12_TJ_A5BB glob 1960 EXP
TOM12_TJ_A5BB glob 1970 Cflx
TOM12_TJ_A5BB glob 1970 PPINT
TOM12_TJ_A5BB glob 1970 sos
TOM12_TJ_A5BB glob 1970 EXP
